In [1]:
from uk.ac.ebi.vfb.neo4j.KB_tools import KB_pattern_writer
import pandas as pd
from ontobio import OntologyFactory
import re
import yaml

#local_test = KB_pattern_writer('http://localhost:7474', 'neo4j', 'neo4j')
#kb = KB_pattern_writer('http://kb.virtualflybrain.org', 'neo4j', 'neo4j')
kbw = KB_pattern_writer('http://kbw.virtualflybrain.org:7474', 'neo4j', 'baNi8HaC')

OF = OntologyFactory()
fbbt = OF.create("/repos/drosophila-anatomy-developmental-ontology/fbbt/releases/fbbt.json")
mapping_table = pd.read_csv("./IndexKey_neuropils_only.tsv", sep='\t')

#template_yaml = yaml.load(yf.read())
#mapping_table.set_index("Index", inplace=True)

mapping_table
# print(template_yaml)
#c = ['label', 'synonyms', 'Index', 'centre','anatomical_type'] 

#[fbbt.label(re.sub('_', ':', x)) for x in mapping_table['Class_ID']]








,Index,Ind_ID,Class_ID,Name,Abbreviation,Note,Centre,Colour
0,1,VFB_00050002,FBbt_00045032,Superior Intermediate Protocerebrum,SIP,NaN,"[498, 129, 227]",78c679
1,2,VFB_00050003,FBbt_00007719,Tritocerebrum dorsal Right,TRd R,NaN,"[602, 409, 231]",88419d
2,3,VFB_00050004,FBbt_00007704,VNC centrolateral Right,VNCcl R,NaN,"[611, 929, 261]",389b64
3,4,VFB_00050005,FBbt_00015028,Mandibula centrolateral Right,MDcl R,NaN,"[638, 450, 254]",ce1256
4,5,VFB_00050006,FBbt_00015035,Maxilla centrolateral Right,MXcl R,NaN,"[642, 484, 255]",e7298a
5,6,VFB_00050007,FBbt_00015021,Labium centrolateral Right,LBcl R,NaN,"[647, 550, 253]",ce1256
6,7,VFB_00050008,FBbt_00015014,Thoracic Segment 1 centrolateral Right,T1cl R,NaN,"[633, 637, 253]",41ab5d
7,8,VFB_00050009,FBbt_00015000,Thoracic Segment 2 centrolateral Right,T2cl R,NaN,"[618, 749, 255]",238443
8,9,VFB_00050010,FBbt_00015007,Thoracic Segment 3 centrolateral Right,T3cl R,NaN,"[605, 881, 256]",41ab5d
9,10,VFB_00050011,FBbt_00015042,Abdominal Segment 1 centrolateral Right,A1cl R,NaN,"[594, 991, 257]",b9563e


In [14]:
ltd = []
for i,r in mapping_table.iterrows():
    LR = ''
    if re.match('.+Left', r.Name): LR = "left "
    if re.match('.+Right', r.Name): LR = "right "
    class_name = fbbt.label(re.sub('_', ':', r.Class_ID))
    label_core = re.sub('(embryonic/larval |larval |segment )', '', class_name)
    d = {}
    d['label'] = LR + label_core + ' on L3 CNS template, Wood2018' # Need L/R!
    d['synonyms'] = [r.Abbreviation, r.Name]
    d['Index'] = r.Index
    d['Anatomical_type'] = r.Class_ID
    d['centre'] = yaml.load(r.Centre)
    d['VFB_id'] = ''
    ltd.append(d)
    
loading_table = pd.DataFrame.from_records(ltd)
loading_table

,Anatomical_type,Index,VFB_id,centre,label,synonyms
0,FBbt_00045032,1,,"[498, 129, 227]",superior intermediate protocerebrum on L3 CNS ...,"[SIP, Superior Intermediate Protocerebrum]"
1,FBbt_00007719,2,,"[602, 409, 231]",right dorsomedial neuropil of tritocerebrum on...,"[TRd R, Tritocerebrum dorsal Right]"
2,FBbt_00007704,3,,"[611, 929, 261]",right centrolateral subdivision of ventral ner...,"[VNCcl R, VNC centrolateral Right]"
3,FBbt_00015028,4,,"[638, 450, 254]",right centrolateral neuropil of mandibular seg...,"[MDcl R, Mandibula centrolateral Right]"
4,FBbt_00015035,5,,"[642, 484, 255]",right centrolateral neuropil of maxillary segm...,"[MXcl R, Maxilla centrolateral Right]"
5,FBbt_00015021,6,,"[647, 550, 253]",right centrolateral neuropil of labial segment...,"[LBcl R, Labium centrolateral Right]"
6,FBbt_00015014,7,,"[633, 637, 253]",right centrolateral neuropil of T1 on L3 CNS t...,"[T1cl R, Thoracic Segment 1 centrolateral Right]"
7,FBbt_00015000,8,,"[618, 749, 255]",right centrolateral neuropil of T2 on L3 CNS t...,"[T2cl R, Thoracic Segment 2 centrolateral Right]"
8,FBbt_00015007,9,,"[605, 881, 256]",right centrolateral neuropil of T3 on L3 CNS t...,"[T3cl R, Thoracic Segment 3 centrolateral Right]"
9,FBbt_00015042,10,,"[594, 991, 257]",right centrolateral neuropil of A1 on L3 CNS t...,"[A1cl R, Abdominal Segment 1 centrolateral Right]"


In [15]:
lt_out = loading_table.copy()

for k,r in lt_out.iterrows():
    synstring = "|".join(r.synonyms)
    lt.out[k, "synonyms"] = synstring
    cstring = "|".join([str(x) for x in r.centre])
    lt_out.at[k, "centre"] = cstring

lt_out.to_csv("loading_table.tsv", sep = "\t")





In [4]:
for i,r in loading_table.iterrows():

    anatomy_attributes = {}
    anatomy_attributes['synonyms'] = r.synonyms
    iris = kbw.add_anatomy_image_set(dataset = "WoodHartenstein2018",
                                     index = r.Index,
                                     imaging_type = "computer graphic", 
                                     label = r.label,
                                     start = 50002, 
                                     template = 'VFBc_00049000', 
                                     anatomical_type=r.Anatomical_type,
                                     anatomy_attributes=anatomy_attributes,
                                     center=r.centre       
                                     )
    r.VFB_id = iris['anatomy']
loading_table.to_csv("Domains_report.tsv")

    

,Anatomical_type,Index,VFB_id,centre,label,synonyms
0,FBbt_00045032,1,,"[498, 129, 227]",superior intermediate protocerebrum on L3 CNS ...,"[SIP, Superior Intermediate Protocerebrum]"
1,FBbt_00007719,2,,"[602, 409, 231]",right dorsomedial neuropil of tritocerebrum on...,"[TRd R, Tritocerebrum dorsal Right]"
2,FBbt_00007704,3,,"[611, 929, 261]",right centrolateral subdivision of ventral ner...,"[VNCcl R, VNC centrolateral Right]"
3,FBbt_00015028,4,,"[638, 450, 254]",right centrolateral neuropil of mandibular seg...,"[MDcl R, Mandibula centrolateral Right]"
4,FBbt_00015035,5,,"[642, 484, 255]",right centrolateral neuropil of maxillary segm...,"[MXcl R, Maxilla centrolateral Right]"
5,FBbt_00015021,6,,"[647, 550, 253]",right centrolateral neuropil of labial segment...,"[LBcl R, Labium centrolateral Right]"
6,FBbt_00015014,7,,"[633, 637, 253]",right centrolateral neuropil of T1 on L3 CNS t...,"[T1cl R, Thoracic Segment 1 centrolateral Right]"
7,FBbt_00015000,8,,"[618, 749, 255]",right centrolateral neuropil of T2 on L3 CNS t...,"[T2cl R, Thoracic Segment 2 centrolateral Right]"
8,FBbt_00015007,9,,"[605, 881, 256]",right centrolateral neuropil of T3 on L3 CNS t...,"[T3cl R, Thoracic Segment 3 centrolateral Right]"
9,FBbt_00015042,10,,"[594, 991, 257]",right centrolateral neuropil of A1 on L3 CNS t...,"[A1cl R, Abdominal Segment 1 centrolateral Right]"


,Anatomical_type,Index,VFB_id,centre,label,synonyms
0,FBbt_00045032,1.0,,498|129|227,superior intermediate protocerebrum on L3 CNS ...,SIP|Superior Intermediate Protocerebrum
1,FBbt_00007719,2.0,,4|9|8|||1|2|9|||2|2|7,right dorsomedial neuropil of tritocerebrum on...,S|I|P|||S|u|p|e|r|i|o|r| |I|n|t|e|r|m|e|d|i|a|...
2,FBbt_00007704,3.0,,6|0|2|||4|0|9|||2|3|1,right centrolateral subdivision of ventral ner...,T|R|d| |R|||T|r|i|t|o|c|e|r|e|b|r|u|m| |d|o|r|...
3,FBbt_00015028,4.0,,6|1|1|||9|2|9|||2|6|1,right centrolateral neuropil of mandibular seg...,V|N|C|c|l| |R|||V|N|C| |c|e|n|t|r|o|l|a|t|e|r|...
4,FBbt_00015035,5.0,,6|3|8|||4|5|0|||2|5|4,right centrolateral neuropil of maxillary segm...,M|D|c|l| |R|||M|a|n|d|i|b|u|l|a| |c|e|n|t|r|o|...
5,FBbt_00015021,6.0,,6|4|2|||4|8|4|||2|5|5,right centrolateral neuropil of labial segment...,M|X|c|l| |R|||M|a|x|i|l|l|a| |c|e|n|t|r|o|l|a|...
6,FBbt_00015014,7.0,,6|4|7|||5|5|0|||2|5|3,right centrolateral neuropil of T1 on L3 CNS t...,L|B|c|l| |R|||L|a|b|i|u|m| |c|e|n|t|r|o|l|a|t|...
7,FBbt_00015000,8.0,,6|3|3|||6|3|7|||2|5|3,right centrolateral neuropil of T2 on L3 CNS t...,T|1|c|l| |R|||T|h|o|r|a|c|i|c| |S|e|g|m|e|n|t|...
8,FBbt_00015007,9.0,,6|1|8|||7|4|9|||2|5|5,right centrolateral neuropil of T3 on L3 CNS t...,T|2|c|l| |R|||T|h|o|r|a|c|i|c| |S|e|g|m|e|n|t|...
9,FBbt_00015042,10.0,,6|0|5|||8|8|1|||2|5|6,right centrolateral neuropil of A1 on L3 CNS t...,T|3|c|l| |R|||T|h|o|r|a|c|i|c| |S|e|g|m|e|n|t|...
